# Notebook 3 — Skin-tone stratified performance via Individual Typology Angle (ITA)**Why this matters.** The paper's central claim is equitable performance onmelanin-rich skin, yet the corpus carries **no Fitzpatrick labels at all**(FACT_CHECK D10/E1). Without some tone estimate there is no measurement behind theclaim. ITA computed on non-lesion skin is the standard retrospective proxy(cf. Corbin & Marques, `ref7`; Bencevic et al., `ref6`).**Method.** Per test image: convert to CIE L\*a\*b\*, segment the lesion by Otsuthreshold on L\*, take the *remaining* (non-lesion) pixels as skin, and compute$$\mathrm{ITA} = \arctan\!\left(\frac{L^{*} - 50}{b^{*}}\right)\cdot\frac{180}{\pi}$$using the **median** over skin pixels. Bin with the standard categories.CPU only, a few minutes. Reuses Notebook 1's cached softmax outputs, so no GPU.**Outputs:** `ita_per_image.csv`, `tone_results.json`, `fig6_tone_performance.pdf`.

In [ ]:
# ============================ EDIT THESE ============================DATA_ROOT  = '/kaggle/input/datasets/jainmanav20/ipd-skin-dataset/IPD_Datasets_Final_Split'CACHE_DIR  = '/kaggle/working'    # where Notebook 1 wrote cache_*.npzOUT_DIR    = '/kaggle/working'# ====================================================================# Notebook 1 MUST have been run first (it writes cache_*_test.npz).import json, timefrom pathlib import Pathfrom collections import Counterimport numpy as npimport pandas as pdfrom PIL import Imageimport cv2import matplotlibmatplotlib.use('Agg')import matplotlib.pyplot as pltfrom sklearn.metrics import accuracy_score, matthews_corrcoefSEED = 42np.random.seed(SEED)DATA_ROOT = Path(DATA_ROOT); CACHE = Path(CACHE_DIR); OUT = Path(OUT_DIR)OUT.mkdir(parents=True, exist_ok=True)CLASSES = ['Basal_Cell_Carcinoma', 'Eczema', 'Keratosis', 'Melanoma', 'Tinea']NUM_CLS = len(CLASSES)# ITA bins, light -> dark (standard categories).# Convention: very light > 55, light 41-55, intermediate 28-41, tan 10-28,# brown -30-10, dark < -30. Intervals are (lo, hi]: a value equal to a boundary# falls in the DARKER bin, so ITA=55 is 'light', not 'very light'.ITA_BINS = [    ('very light',  55,   np.inf),    ('light',       41,   55),    ('intermediate',28,   41),    ('tan',         10,   28),    ('brown',      -30,   10),    ('dark',    -np.inf, -30),]BIN_ORDER = [b[0] for b in ITA_BINS]def ita_bin(ita):    if ita is None or not np.isfinite(ita):        return 'unknown'    for name, lo, hi in ITA_BINS:        if lo < ita <= hi:            return name    return 'dark' if ita <= -30 else 'unknown'

## ITA per imageThe lesion mask is a deliberately simple Otsu threshold on L\*: lesions are usuallydarker than surrounding skin, so the brighter Otsu class is taken as skin. Guardrails: if the "skin" side holds less than 10% of pixels the split is rejected andall pixels are used, and a border-ring fallback is recorded for comparison.

In [ ]:
def compute_ita(path, border_frac=0.15):    # Returns (ita_median_skin, ita_median_border, skin_pixel_frac, ok).    try:        with Image.open(path) as im:            rgb = np.asarray(im.convert('RGB'))    except Exception:        return np.nan, np.nan, np.nan, False    # downscale for speed; ITA is a colour statistic, resolution-insensitive    h, w = rgb.shape[:2]    scale = 256 / max(h, w)    if scale < 1:        rgb = cv2.resize(rgb, (max(1,int(w*scale)), max(1,int(h*scale))),                         interpolation=cv2.INTER_AREA)    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB).astype(np.float64)    # OpenCV 8-bit LAB: L in [0,255] -> [0,100]; a,b in [0,255] with 128 offset    L = lab[..., 0] * 100.0 / 255.0    b = lab[..., 2] - 128.0    # Otsu on L: darker class = lesion, brighter class = skin    Lu8 = np.clip(lab[..., 0], 0, 255).astype(np.uint8)    thr, _ = cv2.threshold(Lu8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)    skin_mask = Lu8 > thr    frac = float(skin_mask.mean())    if frac < 0.10:                      # degenerate split -> use everything        skin_mask = np.ones_like(skin_mask, dtype=bool)        frac = 1.0    def _ita(mask):        Lm, bm = L[mask], b[mask]        valid = np.isfinite(Lm) & np.isfinite(bm) & (np.abs(bm) > 1e-6)        if valid.sum() < 50:            return np.nan        return float(np.median(np.arctan((Lm[valid]-50.0)/bm[valid]) * 180.0/np.pi))    # border ring fallback: outer frame is usually peri-lesional skin    ring = np.zeros(L.shape, dtype=bool)    mh, mw = max(1,int(L.shape[0]*border_frac)), max(1,int(L.shape[1]*border_frac))    ring[:mh,:] = ring[-mh:,:] = ring[:,:mw] = ring[:,-mw:] = True    return _ita(skin_mask), _ita(ring), frac, Truerows = []t0 = time.time()for split in ['train', 'test']:    for cls in CLASSES:        d = DATA_ROOT/split/cls        if not d.is_dir():            continue        for f in sorted(d.iterdir()):            if f.suffix.lower() not in {'.jpg','.jpeg','.png','.bmp','.webp'}:                continue            ita_s, ita_b, frac, ok = compute_ita(f)            rows.append(dict(path=str(f), split=split, cls=cls,                             ita_skin=ita_s, ita_border=ita_b,                             skin_frac=frac, ok=ok, tone_bin=ita_bin(ita_s)))    print(f'{split}: {sum(1 for r in rows if r["split"]==split)} images '          f'({time.time()-t0:.0f}s)')ita_df = pd.DataFrame(rows)ita_df.to_csv(OUT/'ita_per_image.csv', index=False)print(f'\nwrote ita_per_image.csv ({len(ita_df)} rows), failures: {(~ita_df["ok"]).sum()}')print(f'ITA (skin) summary: median={ita_df["ita_skin"].median():.1f} '      f'IQR=({ita_df["ita_skin"].quantile(.25):.1f}, {ita_df["ita_skin"].quantile(.75):.1f})')print(f'skin-mask fraction: median={ita_df["skin_frac"].median():.2f}')

In [ ]:
print('Tone distribution (ITA bin), per split:')dist = (ita_df.groupby(['split','tone_bin']).size().unstack(fill_value=0)        .reindex(columns=BIN_ORDER+['unknown'], fill_value=0))print(dist.to_string())print('\nrow %:')print((dist.div(dist.sum(axis=1), axis=0)*100).round(1).to_string())print('\nTone distribution by class (test only):')tst = ita_df[ita_df['split']=='test']print(tst.groupby(['cls','tone_bin']).size().unstack(fill_value=0)      .reindex(columns=BIN_ORDER+['unknown'], fill_value=0).to_string())print('\nAgreement between lesion-masked and border-ring ITA (sanity on the mask):')both = ita_df.dropna(subset=['ita_skin','ita_border'])if len(both) > 10:    r = np.corrcoef(both['ita_skin'], both['ita_border'])[0,1]    md = float(np.median(np.abs(both['ita_skin']-both['ita_border'])))    print(f'  pearson r={r:.3f} | median |difference| = {md:.1f} deg (n={len(both)})')    print('  A large discrepancy means the Otsu mask is unreliable on this corpus.')

## Per-tone-bin performance, from Notebook 1's cached softmax outputsBins with n < 30 are flagged **UNRELIABLE** and must not be quoted as evidence.

In [ ]:
def load_cache(stem):    p = CACHE/f'cache_{stem}_test.npz'    assert p.exists(), f'missing {p} -- run Notebook 1 first'    z = np.load(p, allow_pickle=False)    return z['probs'], z['labels'], np.array([str(x) for x in z['paths']])probs_E, y, paths_E = load_cache('efficientnet_b1')probs_C, _, paths_C = load_cache('convnext_tiny')probs_S, _, paths_S = load_cache('swin_tiny')assert list(paths_E) == list(paths_C) == list(paths_S), 'cache path order differs'probs_ECS = (probs_E + probs_C + probs_S) / 3.0print(f'loaded caches: n={len(y)}')# align ITA rows to the cached test orderita_test = ita_df[ita_df['split']=='test'].set_index('path')missing = [p for p in paths_E if p not in ita_test.index]assert not missing, f'{len(missing)} cached test paths lack an ITA row, e.g. {missing[:3]}'tone = np.array([ita_test.loc[p,'tone_bin'] for p in paths_E])ita_v = np.array([ita_test.loc[p,'ita_skin'] for p in paths_E], dtype=float)CONFIGS = {'EfficientNet-B1': probs_E, 'ConvNeXt-Tiny': probs_C,           'Swin-Tiny': probs_S, 'Ensemble (E+C+S)': probs_ECS}N_BOOT = 2000rng = np.random.default_rng(SEED)def boot_ci_acc(y_sub, pred_sub, nboot=N_BOOT):    if len(y_sub) == 0:        return (float('nan'), float('nan'))    idx = rng.integers(0, len(y_sub), size=(nboot, len(y_sub)))    accs = (y_sub[idx] == pred_sub[idx]).mean(axis=1)    return float(np.percentile(accs, 2.5)), float(np.percentile(accs, 97.5))TONE_RESULTS = {}present = [b for b in BIN_ORDER+['unknown'] if (tone == b).sum() > 0]for name, probs in CONFIGS.items():    pred = probs.argmax(1)    TONE_RESULTS[name] = {}    print(f'\n=== {name} ===')    print(f'{"tone bin":14s} {"n":>5} {"acc":>8} {"95% CI":>18} {"flag":>12}')    print('-'*62)    for b in present:        m = tone == b        ys, ps = y[m], pred[m]        acc = float((ys == ps).mean())        lo, hi = boot_ci_acc(ys, ps)        unrel = int(m.sum()) < 30        rec = dict(n=int(m.sum()), accuracy=acc, ci_lo=lo, ci_hi=hi,                   unreliable_n_lt_30=unrel,                   per_class_recall={}, class_support={})        for i, cls in enumerate(CLASSES):            cm_ = m & (y == i)            rec['class_support'][cls] = int(cm_.sum())            rec['per_class_recall'][cls] = (float((pred[cm_] == i).mean())                                            if cm_.sum() else None)        TONE_RESULTS[name][b] = rec        print(f'{b:14s} {m.sum():>5} {acc*100:>7.2f}% '              f'[{lo*100:>6.2f},{hi*100:>6.2f}] {"UNRELIABLE" if unrel else "":>12}')# worst-vs-best gap on reliable bins only, for the ensemblerel = {b: r for b, r in TONE_RESULTS['Ensemble (E+C+S)'].items()       if not r['unreliable_n_lt_30'] and b != 'unknown'}GAP = Noneif len(rel) >= 2:    best = max(rel.items(), key=lambda kv: kv[1]['accuracy'])    worst = min(rel.items(), key=lambda kv: kv[1]['accuracy'])    GAP = dict(best_bin=best[0], best_acc=best[1]['accuracy'],               worst_bin=worst[0], worst_acc=worst[1]['accuracy'],               gap_pp=(best[1]['accuracy']-worst[1]['accuracy'])*100)    print(f'\nEnsemble tone gap (reliable bins only): '          f'{GAP["best_bin"]} {GAP["best_acc"]*100:.2f}% vs '          f'{GAP["worst_bin"]} {GAP["worst_acc"]*100:.2f}%  '          f'=> {GAP["gap_pp"]:.2f} pp')else:    print('\nFewer than 2 reliable tone bins -> no defensible gap estimate.')

In [ ]:
with open(OUT/'tone_results.json','w',encoding='utf-8') as f:    json.dump(dict(        meta=dict(notebook='03_skin_tone_ita.ipynb', seed=SEED, n_boot=N_BOOT,                  method='ITA on non-lesion (Otsu-brighter) pixels, median',                  ita_bins={b[0]: [None if not np.isfinite(b[1]) else b[1],                                   None if not np.isfinite(b[2]) else b[2]]                            for b in ITA_BINS},                  proxy_warning='ITA is a PROXY. No ground-truth Fitzpatrick labels exist '                                'for this corpus. Do not report as Fitzpatrick type.'),        tone_distribution={s: {b: int(((ita_df['split']==s)&(ita_df['tone_bin']==b)).sum())                               for b in BIN_ORDER+['unknown']}                           for s in ['train','test']},        per_config=TONE_RESULTS,        ensemble_tone_gap=GAP,    ), f, indent=2)print('wrote tone_results.json')# figure (every value also in the JSON/CSV)fig, axes = plt.subplots(1, 2, figsize=(15, 5))ax = axes[0]counts = [int((tone == b).sum()) for b in present]ax.bar(range(len(present)), counts, color='#4C72B0', edgecolor='black', linewidth=0.6)for i, c in enumerate(counts):    ax.text(i, c, str(c), ha='center', va='bottom', fontsize=8, fontweight='bold')ax.set_xticks(range(len(present))); ax.set_xticklabels(present, rotation=25, ha='right', fontsize=8.5)ax.set_ylabel('test images'); ax.set_title('ITA tone distribution (test)', fontweight='bold')ax.axhline(30, color='red', ls='--', lw=1, label='n = 30 reliability floor'); ax.legend(fontsize=8)ax = axes[1]width = 0.2for k, (name, _) in enumerate(CONFIGS.items()):    xs, ys_, los, his = [], [], [], []    for i, b in enumerate(present):        r = TONE_RESULTS[name][b]        if r['n'] == 0: continue        xs.append(i + (k-1.5)*width); ys_.append(r['accuracy']*100)        los.append((r['accuracy']-r['ci_lo'])*100); his.append((r['ci_hi']-r['accuracy'])*100)    ax.bar(xs, ys_, width=width, label=name, edgecolor='black', linewidth=0.4,           yerr=[los, his], capsize=2, error_kw=dict(lw=0.7))ax.set_xticks(range(len(present))); ax.set_xticklabels(present, rotation=25, ha='right', fontsize=8.5)ax.set_ylabel('Accuracy (%)'); ax.set_ylim(0, 105)ax.set_title('Accuracy by ITA tone bin (95% CI)', fontweight='bold'); ax.legend(fontsize=7.5)plt.suptitle('Skin-tone stratified performance (ITA proxy — not ground-truth Fitzpatrick)',             fontsize=12, fontweight='bold')plt.tight_layout(); plt.savefig(OUT/'fig6_tone_performance.pdf', bbox_inches='tight'); plt.close()print('wrote fig6_tone_performance.pdf')

## Limitations of ITA as a Fitzpatrick proxy — state these in the paper1. **It is not Fitzpatrick type.** ITA measures apparent colour in one photograph;   Fitzpatrick is a self-reported/clinical phototype describing UV response. The   mapping between them is approximate and was validated on controlled photography,   not on heterogeneous web-sourced images.2. **Illumination dominates.** These images come from five sources with no colour   calibration, no fixed white balance and no colour chart. Lighting and camera   processing can shift ITA by more than the width of a tone bin, so bin assignment   carries unquantified error.3. **Dermoscopy vs. clinical.** The Melanoma Cancer Image Dataset contributes   dermoscopic images, which use contact optics, immersion fluid and their own   illuminant. ITA is not comparable between dermoscopic and clinical photographs,   and tone bins will partly encode *modality*, not skin tone.4. **The lesion mask is crude.** A single Otsu threshold on L\* is not a trained   segmenter. On large, light or low-contrast lesions the "skin" region may include   lesion pixels, biasing ITA. The border-ring cross-check in this notebook   quantifies, but does not fix, this.5. **No ground truth exists.** Nothing in the corpus lets us validate these   estimates, so per-bin numbers cannot be checked for calibration — only reported   as a proxy.6. **Confounding with class.** If a source contributes mostly one class *and* one   tone range, per-tone accuracy partly reflects per-source difficulty. The   class-by-tone table above is the diagnostic for this.7. **Small bins.** Any bin with n < 30 is flagged unreliable; its CI will be too   wide to support a fairness claim in either direction.**Conclusion for the paper:** report this as an exploratory proxy analysis withthese caveats stated, and do *not* phrase it as a Fitzpatrick-stratified result.A defensible equity claim needs tone labels collected with the images.